# 🕵️ Dia 29 — Registro de domínios: WHOIS, RDAP e histórico de registro

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 29: Domain registration: WHOIS, RDAP, and registration history* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-29.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sexta, 9 de outubro (a hospedagem do site falso acaba de ser suspensa). O comunicado foi editado na noite de 4/out. Rafa quer saber desde quando o palco estava montado: *"Domínio não nasce na véspera por acaso. RDAP, WHOIS, e o histórico do que der. E data cada linha."*

Neste notebook você vai:

1. **Consultar RDAP de verdade** (Registro.br e rdap.org) com `urllib`, com plano B offline.
2. **Ler o RDAP da Operação Isca** e transformar `events`, `status` e `entities` numa tabela.
3. **Medir a distância** entre cada registro e o primeiro SMS.
4. **Gerar o Gantt** em Mermaid, com a fonte de cada linha.
5. **Jogar o quiz dos status EPP**.

🐍 Só biblioteca padrão.

## 1. RDAP ao vivo (com plano B)

O RDAP é JSON por HTTPS. Para `.br`, o servidor é `https://rdap.registro.br/`; para gTLDs, o `rdap.org` redireciona para o registry certo (o `urllib` segue redirecionamentos sozinho, como o `curl -L`).

Se a rede falhar (Colab sem saída, proxy, servidor fora), usamos uma **amostra embutida** com a mesma estrutura, marcada como tal.

In [ ]:
import json, urllib.request
from datetime import datetime, timezone

# Amostra embutida: estrutura de uma resposta RDAP, com valores ILUSTRATIVOS.
# Ela só existe para o notebook rodar sem rede; nunca a cite como fato.
AMOSTRA = {
    "ldhName": "exemplo-ilustrativo.br",
    "status": ["active"],
    "events": [{"eventAction": "registration", "eventDate": "1999-01-01T00:00:00Z"},
               {"eventAction": "last changed", "eventDate": "2026-01-01T00:00:00Z"}],
    "nameservers": [{"ldhName": "a.ns.exemplo-ilustrativo.br"}],
}


def consultar_rdap(url: str):
    """Tenta o RDAP real; devolve (json, fonte, hora_utc_da_consulta)."""
    hora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")   # registre SEMPRE a hora
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/rdap+json",
                                                   "User-Agent": "estudo-investigador-cyber/1.0"})
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.load(r), url, hora
    except Exception as e:                                           # sem rede ou erro HTTP
        return AMOSTRA, f"AMOSTRA EMBUTIDA (falha: {type(e).__name__})", hora


for url in ("https://rdap.registro.br/domain/registro.br",
            "https://rdap.org/domain/example.com"):
    dados, fonte, hora = consultar_rdap(url)
    eventos = {e["eventAction"]: e["eventDate"] for e in dados.get("events", [])}
    print(f"Fonte: {fonte} | consultado em {hora}")
    print("   domínio:", dados.get("ldhName"), "| status:", dados.get("status"))
    print("   eventos:", eventos)
    print("   NS:", [n.get("ldhName") for n in dados.get("nameservers", [])], "\n")

In [ ]:
# OPCIONAL: os comandos do original e o caminho .br (o teste pula linhas com !)
!apt-get -qq install -y whois jq > /dev/null
!whois -h whois.iana.org br | grep -i '^whois'
!whois -h whois.registro.br registro.br | head -25
!curl -s https://rdap.registro.br/domain/registro.br | jq -r '.events[] | "\(.eventAction)\t\(.eventDate)"'
print("(fora do Colab, as linhas com ! foram puladas)")

## 2. O RDAP da Operação Isca

Os domínios `.example` não existem na internet. As respostas abaixo seguem **o formato RDAP real** e estão em `caso-continuo/dados/dia-29/rdap-isca.json`. Tudo fictício.

In [ ]:
RDAP_ISCA = json.loads(r"""{"serraazul.example": {"objectClassName": "domain", "ldhName": "serraazul.example", "status": ["client transfer prohibited", "client delete prohibited"], "events": [{"eventAction": "registration", "eventDate": "2004-03-15T13:00:00Z"}, {"eventAction": "expiration", "eventDate": "2027-03-15T13:00:00Z"}, {"eventAction": "last changed", "eventDate": "2025-11-20T10:12:00Z"}], "nameservers": [{"objectClassName": "nameserver", "ldhName": "ns1.provedor-dns.example"}], "secureDNS": {"delegationSigned": true}, "entities": [{"objectClassName": "entity", "roles": ["registrar"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Registrar Tradicional (fictício)"]]]}, {"objectClassName": "entity", "roles": ["registrant"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Cooperativa de Crédito Serra Azul (fictícia)"]]]}], "notices": [{"title": "Dado fictício", "description": ["Operação Isca — material de estudo. Nada aqui é real."]}]}, "serraazul-pix.example": {"objectClassName": "domain", "ldhName": "serraazul-pix.example", "status": ["client transfer prohibited"], "events": [{"eventAction": "registration", "eventDate": "2026-10-02T14:22:10Z"}, {"eventAction": "expiration", "eventDate": "2027-10-02T14:22:10Z"}, {"eventAction": "last changed", "eventDate": "2026-10-02T14:22:10Z"}], "nameservers": [{"objectClassName": "nameserver", "ldhName": "ns1.dnsbarato.example"}, {"objectClassName": "nameserver", "ldhName": "ns2.dnsbarato.example"}], "secureDNS": {"delegationSigned": false}, "entities": [{"objectClassName": "entity", "roles": ["registrar"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Registrar Barato (fictício)"]]]}, {"objectClassName": "entity", "roles": ["registrant"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "REDACTED FOR PRIVACY"]]]}], "notices": [{"title": "Dado fictício", "description": ["Operação Isca — material de estudo. Nada aqui é real."]}]}, "serraazul-seguranca.example": {"objectClassName": "domain", "ldhName": "serraazul-seguranca.example", "status": ["client transfer prohibited"], "events": [{"eventAction": "registration", "eventDate": "2026-10-02T14:25:47Z"}, {"eventAction": "expiration", "eventDate": "2027-10-02T14:25:47Z"}, {"eventAction": "last changed", "eventDate": "2026-10-02T14:25:47Z"}], "nameservers": [{"objectClassName": "nameserver", "ldhName": "ns1.dnsbarato.example"}], "secureDNS": {"delegationSigned": false}, "entities": [{"objectClassName": "entity", "roles": ["registrar"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Registrar Barato (fictício)"]]]}, {"objectClassName": "entity", "roles": ["registrant"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "REDACTED FOR PRIVACY"]]]}], "notices": [{"title": "Dado fictício", "description": ["Operação Isca — material de estudo. Nada aqui é real."]}]}, "coopserra-azul.example": {"objectClassName": "domain", "ldhName": "coopserra-azul.example", "status": ["client transfer prohibited"], "events": [{"eventAction": "registration", "eventDate": "2026-10-03T02:03:00Z"}, {"eventAction": "expiration", "eventDate": "2027-10-03T02:03:00Z"}, {"eventAction": "last changed", "eventDate": "2026-10-03T02:03:00Z"}], "nameservers": [{"objectClassName": "nameserver", "ldhName": "ns1.registrar-parking.example"}], "secureDNS": {"delegationSigned": false}, "entities": [{"objectClassName": "entity", "roles": ["registrar"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Registrar Parking (fictício)"]]]}, {"objectClassName": "entity", "roles": ["registrant"], "vcardArray": ["vcard", [["version", {}, "text", "4.0"], ["fn", {}, "text", "Privacy Proxy Exemplo (fictício)"]]]}], "notices": [{"title": "Dado fictício", "description": ["Operação Isca — material de estudo. Nada aqui é real."]}]}}""")


def resumo(nome: str, r: dict) -> dict:
    """Extrai os campos com peso investigativo (Passo 3 do original)."""
    ev = {e["eventAction"]: e["eventDate"] for e in r["events"]}
    def papel(p):                                  # entities: registrar, registrant...
        for ent in r["entities"]:
            if p in ent["roles"]:
                return ent["vcardArray"][1][1][3]  # o campo "fn" do vCard
        return "-"
    return {
        "dominio": nome,
        "criado": ev.get("registration"),
        "expira": ev.get("expiration"),
        "alterado": ev.get("last changed"),
        "registrar": papel("registrar"),
        "titular": papel("registrant"),
        "ns": ", ".join(n["ldhName"] for n in r["nameservers"]),
        "status": ", ".join(r["status"]),
        "dnssec": r["secureDNS"]["delegationSigned"],
    }


tabela = [resumo(n, r) for n, r in RDAP_ISCA.items()]
for t in tabela:
    print(f"\n▶ {t['dominio']}")
    for k, v in t.items():
        if k != "dominio":
            print(f"   {k:10s} {v}")

## 3. Quanto tempo antes do golpe?

O primeiro SMS saiu às **07h12 BRT de 5/out/2026** (fato fixo do caso). BRT é UTC-3.

In [ ]:
from datetime import timedelta

BRT = timezone(timedelta(hours=-3))
PRIMEIRO_SMS = datetime(2026, 10, 5, 7, 12, tzinfo=BRT)


def iso(s):                                                   # "2026-10-02T14:22:10Z" -> datetime
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


print(f"Primeiro SMS: {PRIMEIRO_SMS.astimezone(timezone.utc):%Y-%m-%d %H:%M} UTC\n")
for t in tabela:
    delta = PRIMEIRO_SMS - iso(t["criado"])
    dias, resto = delta.days, delta.seconds // 3600
    quando = "antes" if delta.total_seconds() > 0 else "depois"
    print(f"{t['dominio']:30s} criado {t['criado']}  ->  {abs(dias)} d {resto} h {quando} do 1º SMS")

pix, seg = iso(RDAP_ISCA["serraazul-pix.example"]["events"][0]["eventDate"]), \
           iso(RDAP_ISCA["serraazul-seguranca.example"]["events"][0]["eventDate"])
print(f"\nIntervalo entre os dois imitadores: {(seg - pix).seconds // 60} min {(seg - pix).seconds % 60} s")
print("Mesmo registrar, mesmo dia, minutos de diferença: aresta 'registro coordenado' (provável, 1 fonte).")

## 4. O histórico (e a armadilha do mesmo nome)

Serviços de WHOIS histórico mostram estados antigos. O arquivo do caso traz retratos **fictícios** de um serviço também fictício. Cada linha tem data: um retrato é o que o serviço viu naquele dia.

In [ ]:
import csv, io

HISTORICO = list(csv.DictReader(io.StringIO("""dominio,data_snapshot,fonte,registrar,ns,titular
serraazul.example,2012-05-02,histórico (serviço fictício),Registrar Antigo (fictício),ns1.hospedagem-antiga.example,Cooperativa de Crédito Serra Azul
serraazul.example,2016-07-19,histórico (serviço fictício),Registrar Tradicional (fictício),ns1.hospedagem-antiga.example,Cooperativa de Crédito Serra Azul
serraazul.example,2021-02-03,histórico (serviço fictício),Registrar Tradicional (fictício),ns1.provedor-dns.example,REDACTED FOR PRIVACY
serraazul-pix.example,2026-10-07,histórico (serviço fictício),Registrar Barato (fictício),ns1.dnsbarato.example,REDACTED FOR PRIVACY
coopserra-azul.example,2019-04-10,histórico (serviço fictício),Registrar Antigo (fictício),ns1.parking.example,"Associação de Feirantes Serra Azul (fictícia, sem relação com o caso)"
coopserra-azul.example,2024-05-02,histórico (serviço fictício),Registrar Antigo (fictício),ns1.parking.example,status: pendingDelete (expirou sem renovação)
""")))

for h in HISTORICO:
    print(f"{h['data_snapshot']}  {h['dominio']:24s} {h['registrar']:34s} titular: {h['titular']}")

print("\n⚠️ coopserra-azul.example: o ID de analytics G-ISCA7K2Q9P (Dias 19-26) o liga ao caso HOJE.")
print("   Mas entre 2019 e 2024 o nome era de OUTRO titular, sem relação com o caso,")
print("   e passou por pendingDelete. O registro de 03/10/2026 é um novo registro (outra 'vida' do nome).")
print("   Histórico de um titular anterior NÃO é aresta para o caso atual.")

## 5. O Gantt da Operação Isca

O código abaixo gera o Mermaid do Passo 5. Cada linha diz se veio do **registry** (RDAP ao vivo) ou do **histórico**. Cole o resultado no seu `notas/` (o GitHub renderiza) ou em [mermaid.live](https://mermaid.live).

In [ ]:
linhas = ["gantt", "    title Linha do tempo de registro, Operação Isca (fictício)",
          "    dateFormat YYYY-MM-DD"]

for t in tabela:
    nome = t["dominio"]
    linhas.append(f"    section {nome}")
    criado = t["criado"][:10]
    linhas.append(f"    Registrado (registry)        :milestone, {criado}, 0d")
    # histórico desse domínio, se houver
    for h in HISTORICO:
        if h["dominio"] == nome:
            rotulo = h["registrar"].replace(" (fictício)", "")      # rótulo curto para o gráfico
            linhas.append(f"    {rotulo} (histórico) :milestone, {h['data_snapshot']}, 0d")
    if t["alterado"] and t["alterado"][:10] != criado:
        linhas.append(f"    Alterado (registry)          :milestone, {t['alterado'][:10]}, 0d")

linhas += ["    section Campanha",
           "    PDF editado (Dia 28)         :milestone, 2026-10-04, 0d",
           "    SMS disparados               :milestone, 2026-10-05, 0d"]

gantt = "\n".join(linhas)
print("```mermaid\n" + gantt + "\n```")
open("gantt-isca.md", "w", encoding="utf-8").write("```mermaid\n" + gantt + "\n```\n")
print("\n💾 salvo em gantt-isca.md")

> 💡 O `serraazul.example` começa em 2004, então o gráfico fica comprido. Para a reunião com a Marta, faça uma versão só de set–out/2026: a história do golpe cabe em duas semanas.

## 6. 🎮 Quiz dos status EPP

Para cada status, escolha o significado: **1** trava de rotina · **2** registrado mas sem resolver (hold) · **3** prestes a ser liberado. `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
QUIZ = [
    ("clientTransferProhibited", "1", "Trava de rotina contra transferência não autorizada."),
    ("serverHold", "2", "O registry tirou da zona: registrado, mas não resolve (abuso, ordem, disputa)."),
    ("pendingDelete", "3", "Fim do ciclo: vai ser liberado e alguém pode registrar de novo."),
    ("clientHold", "2", "O registrar suspendeu a resolução (abuso, falta de pagamento)."),
    ("client delete prohibited", "1", "É o mesmo clientDeleteProhibited, escrito em palavras pelo RDAP."),
    ("clientUpdateProhibited", "1", "Trava de rotina contra alteração."),
    ("redemptionPeriod", "3", "Expirou e foi apagado; ainda dá para o antigo titular recuperar, depois vem o pendingDelete."),
    ("server hold (no RDAP)", "2", "RDAP escreve em palavras; mesmo significado do serverHold."),
]

MODO_JOGO = True
acertos = 0
for i, (status, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {status}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1 rotina | 2 hold | 3 liberação: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 6.")

## 7. 📝 A frase do Passo 6

O original pede uma frase situando o registro do imitador em relação a eventos públicos da organização. Na Isca, o evento é o próprio golpe (e o comunicado de fim de ano que ele reaproveitou).

In [ ]:
#@title Frase de contexto { display-mode: "form" }
frase = "serraazul-pix.example foi registrado em 02/10/2026 14h22 UTC, menos de 3 dias antes do 1º SMS; serraazul-seguranca.example 3 min depois, mesmo registrar."  #@param {type:"string"}
inferencia = "Provável preparação coordenada da campanha (inferência; contradiria: registros por titulares distintos ou um dos domínios sem uso no golpe)."  #@param {type:"string"}
fonte = "RDAP (registry), consultado na data da coleta; histórico: serviço X em DD/MM"  #@param {type:"string"}

nota = f"- Fato: {frase}\n- Inferência (rotulada): {inferencia}\n- Fonte: {fonte}\n"
print(nota)
open("contexto-registro.md", "w", encoding="utf-8").write(nota)
try:
    from google.colab import files
    files.download("gantt-isca.md"); files.download("contexto-registro.md")
except ImportError:
    print("(Fora do Colab: arquivos na pasta atual.)")

## ✅ Antes de fechar o Dia 29

- [ ] RDAP real consultado (ou plano B registrado como tal), com hora UTC
- [ ] Tabela dos quatro domínios da Isca, com fonte por linha
- [ ] Distância registro → SMS calculada
- [ ] Armadilha do `coopserra-azul.example` explicada nas notas
- [ ] Gantt gerado; fato e inferência separados
- [ ] Quiz com pelo menos 6/8

**Amanhã (Dia 30):** DNS e Certificate Transparency. O certificado do site falso cobre mais de um nome, e a infraestrutura vira um grafo.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>